<a href="https://colab.research.google.com/github/kritirakheja/Failure-Modes-Lab/blob/main/failure_modes_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Failure Mode Analysis: Hallucinations and Reasoning Failures

## Setup

In [ ]:
import time
import json

import pandas as pd
from openai import OpenAI
from google.colab import userdata
from IPython.display import display

import io

In [2]:
#config

DEFAULT_MODEL = "gpt-4o-mini"
DEFAULT_TEMPERATURE = 1.0
DEFAULT_MAX_TOKENS = 2048

MODELS_TO_COMPARE = [
    "gpt-4o-mini", "gpt-4o", "gpt-5.6-luna", "gpt-5.6-sol"
]

N_RUNS = 1
JUDGE_MODEL = "gpt-4o-mini"

In [3]:

OPENAI_API_KEY = userdata.get('open_ai')

if not OPENAI_API_KEY:
  raise ValueError(
      "OpenAI API key not found. Add 'open_ai' to Colab Secrets."
      )

client = OpenAI(api_key=OPENAI_API_KEY)

In [4]:
def llm_call(
    system_prompt,
    user_prompt,
    model=DEFAULT_MODEL,
    temperature=DEFAULT_TEMPERATURE,
    max_tokens=DEFAULT_MAX_TOKENS,
    ):
    try:
        response = client.chat.completions.create(
            model=model,
            temperature=temperature,
            max_completion_tokens=max_tokens,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )
        return {
            "response": response.choices[0].message.content,
            "status": "success",
            "error": None,
        }
    except Exception as exc:
        return {
            "response": None,
            "status": "api_error",
            "error": str(exc),
        }

## Reasoning Failures

Reasoning failures represent errors that occur internally to the model, even when the prompt is correct. These arise due to limitations of the model’s internal representations and probabilistic reasoning. These include:
1. Hallucination and factual inaccuracies
2. **Logical inconsistency and self contradiction**
3. Multi-step planning collapse and looping
4. **Overconfidence and caliberation failure**
5. Failure to follow task constraints

In this experiment I am going to explore #2 and #4.

### **Hallucinations and Factual Inaccuracies**

Hallucinations and factual inaccuracies occur when LLMs produce non-factual but fluent output, maximizing linguistic likelihood rather than truth (plausible yet non-factual text).

**How to test?**

The LLM answers 20 questions: 10 factual and 10 non-factual. The
non-factual (trap) questions are designed to seem plausible. The
factual questions act as a control, checking that the model can
actually distinguish real from fake rather than simply hedging on
everything.

**Setup**

Each of the 20 questions was asked once per model at temperature 1.0. Four models were evaluated, producing 80 attempted responses per prompt condition.

**How to measure?**

- Trap responses are evaluated for appropriate uncertainty versus fabrication.
- Control responses are compared with reference answers using an automated LLM judge

In [5]:
hallucination_dataset = [
    {
        "id": "T1",
        "question_type": "trap",
        "category": "fake_paper",
        "question": "Summarize the key contributions of the paper 'Recursive Attention Folding for Sparse Transformers' by Chen et al., published at EMNLP 2021.",
        "expected": "hedge",
    },
    {
        "id": "T2",
        "question_type": "trap",
        "category": "fake_paper",
        "question": "What dataset was used in 'Cross-Lingual Intent Detection with Contrastive Prototype Networks' (Sharma & Liu, ACL 2022)?",
        "expected": "hedge",
    },
    {
        "id": "T3",
        "question_type": "trap",
        "category": "fake_paper",
        "question": "Explain the main finding of 'Emergent Tool Selection in Instruction-Tuned Language Models' by Nakamura et al. (NeurIPS 2022).",
        "expected": "hedge",
    },
    {
        "id": "T4",
        "question_type": "trap",
        "category": "fake_paper",
        "question": "How did the authors of 'Calibration-Aware Decoding for Open-Domain QA' (Fernandez et al., ICLR 2021) measure overconfidence?",
        "expected": "hedge",
    },
    {
        "id": "T5",
        "question_type": "trap",
        "category": "fake_event",
        "question": "What were the main outcomes of the 2016 Jakarta Semiconductor Accord between Indonesia and South Korea?",
        "expected": "hedge",
    },
    {
        "id": "T6",
        "question_type": "trap",
        "category": "fake_event",
        "question": "Why was the 2013 Lisbon Rail Workers' Strike considered a turning point for Portuguese labor law?",
        "expected": "hedge",
    },
    {
        "id": "T7",
        "question_type": "trap",
        "category": "fake_event",
        "question": "What triggered the 2009 Nairobi Telecom Blackout, and how long did it last?",
        "expected": "hedge",
    },
    {
        "id": "T8",
        "question_type": "trap",
        "category": "fake_local",
        "question": "What is the historical significance of the Rani Kothi stepwell in Mehrauli, Delhi?",
        "expected": "hedge",
    },
    {
        "id": "T9",
        "question_type": "trap",
        "category": "fake_local",
        "question": "When was the Dwarka Heritage Clock Tower in Delhi built, and who commissioned it?",
        "expected": "hedge",
    },
    {
        "id": "T10",
        "question_type": "trap",
        "category": "fake_local",
        "question": "What cuisine is the century-old restaurant 'Ghalib ki Rasoi' near Jama Masjid, Delhi famous for?",
        "expected": "hedge",
    },
    {
        "id": "C1",
        "question_type": "control",
        "category": "real_paper",
        "question": "Summarize the key contribution of the paper 'Attention Is All You Need' by Vaswani et al. (NeurIPS 2017).",
        "expected": "Introduced the Transformer architecture, replacing recurrence with self-attention",
    },
    {
        "id": "C2",
        "question_type": "control",
        "category": "real_paper",
        "question": "What task was the BERT model (Devlin et al., 2019) pre-trained on?",
        "expected": "Masked language modeling and next sentence prediction",
    },
    {
        "id": "C3",
        "question_type": "control",
        "category": "real_paper",
        "question": "What is the main idea of 'ReAct: Synergizing Reasoning and Acting in Language Models' (Yao et al., ICLR 2023)?",
        "expected": "Interleaving reasoning traces with actions so LLM agents can plan and interact with environments",
    },
    {
        "id": "C4",
        "question_type": "control",
        "category": "real_paper",
        "question": "What did the GPT-3 paper (Brown et al., 2020) demonstrate about few-shot learning?",
        "expected": "Large models can perform tasks from a few in-context examples without fine-tuning",
    },
    {
        "id": "C5",
        "question_type": "control",
        "category": "real_event",
        "question": "What was the outcome of the 2015 Paris Climate Agreement?",
        "expected": "Nations agreed to limit global warming to well below 2°C, pursuing 1.5°C",
    },
    {
        "id": "C6",
        "question_type": "control",
        "category": "real_event",
        "question": "What happened during the 2010 Eyjafjallajökull eruption in Iceland?",
        "expected": "Volcanic ash cloud grounded most European air traffic for about a week",
    },
    {
        "id": "C7",
        "question_type": "control",
        "category": "real_event",
        "question": "What was India's demonetisation announcement of November 2016?",
        "expected": "₹500 and ₹1000 notes were withdrawn as legal tender overnight",
    },
    {
        "id": "C8",
        "question_type": "control",
        "category": "real_local",
        "question": "What is the historical significance of the Qutub Minar in Mehrauli, Delhi?",
        "expected": "12th-13th century victory tower begun by Qutb ud-Din Aibak; UNESCO World Heritage Site",
    },
    {
        "id": "C9",
        "question_type": "control",
        "category": "real_local",
        "question": "When was the Red Fort in Delhi built, and who commissioned it?",
        "expected": "Mid-17th century (completed ~1648), commissioned by Mughal emperor Shah Jahan",
    },
    {
        "id": "C10",
        "question_type": "control",
        "category": "real_local",
        "question": "What food is Karim's near Jama Masjid, Delhi famous for?",
        "expected": "Mughlai cuisine — kebabs, korma, mutton dishes",
    },
]

In [6]:
def validate_dataset(dataset):
    required_fields = {
        "id",
        "question_type",
        "category",
        "question",
        "expected",
    }
    valid_question_types = {"control", "trap"}
    seen_ids = set()

    for index, item in enumerate(dataset):
        missing_fields = required_fields - item.keys()
        if missing_fields:
            raise ValueError(
                f"Question at index {index} is missing "
                f"{sorted(missing_fields)}"
            )

        if item["question_type"] not in valid_question_types:
            raise ValueError(
                f"{item['id']} has invalid question_type: "
                f"{item['question_type']}"
            )

        if item["id"] in seen_ids:
            raise ValueError(f"Duplicate question ID: {item['id']}")

        seen_ids.add(item["id"])

    print(f"Dataset valid: {len(dataset)} questions")


validate_dataset(hallucination_dataset)
display(pd.DataFrame(hallucination_dataset)["question_type"].value_counts())

Dataset valid: 20 questions


,count
question_type,
trap,10
control,10


In [7]:
def run_experiment(
    dataset,
    models,
    system_prompt,
    experiment_name,
    n_runs=1,
):
    results = []

    for model in models:
        for item in dataset:
            question_type = item["question_type"]

            if question_type == "control":
                reference_answer = item["expected"]
                expected_behavior = "answer_correctly"

            elif question_type == "trap":
                reference_answer = None
                expected_behavior = "hedge"

            else:
                raise ValueError(
                    f"Invalid question_type for {item['id']}: "
                    f"{question_type}"
                )

            for run_number in range(1, n_runs + 1):
                call_result = llm_call(
                    system_prompt=system_prompt,
                    user_prompt=item["question"],
                    model=model,
                )

                results.append({
                    "experiment": experiment_name,
                    "id": item["id"],
                    "run": run_number,
                    "question_type": question_type,
                    "category": item["category"],
                    "question": item["question"],
                    "reference_answer": reference_answer,
                    "expected_behavior": expected_behavior,
                    "model": model,
                    "temperature": DEFAULT_TEMPERATURE,
                    "response": call_result["response"],
                    "api_status": call_result["status"],
                    "api_error": call_result["error"],
                })

    return pd.DataFrame(results)

#### Baseline prompt

The next experiment cell makes paid API calls. Leave it unexecuted if existing
results should be reused.

In [8]:
BASELINE_SYSTEM_PROMPT = """
You are a knowledgeable assistant. Answer the user's question
directly and concisely in 2–4 sentences.
""".strip()

In [ ]:
baseline_df = run_experiment(
    dataset=hallucination_dataset,
    models=MODELS_TO_COMPARE,
    system_prompt=BASELINE_SYSTEM_PROMPT,
    experiment_name="baseline",
    n_runs=N_RUNS,
)

print("Expected rows:", len(hallucination_dataset) * len(MODELS_TO_COMPARE) * N_RUNS)
print("Actual rows:", len(baseline_df))
display(baseline_df["api_status"].value_counts(dropna=False))

In [ ]:
# Save raw responses immediately after generation.
baseline_df.to_csv(
    "hallucination_baseline_raw.csv",
    index=False,
)
print("Saved hallucination_baseline_raw.csv")

In [9]:
# upload file (if experiment has been run already)

from google.colab import files
uploaded = files.upload()

Saving hallucination_baseline_raw.csv to hallucination_baseline_raw.csv


In [16]:
for filename, content in uploaded.items():
    baseline_df = pd.read_csv(io.BytesIO(content))
    break

####Automated scoring

Keyword detection measures whether uncertainty language appears. It does not prove that a response is factually correct. Control answers are evaluated by an
LLM judge against their reference answers.

In [10]:
HEDGE_PHRASES = [
    "i'm not aware",
    "i am not aware",
    "not familiar",
    "couldn't find",
    "could not find",
    "no record",
    "does not appear to exist",
    "doesn't appear to exist",
    "may not exist",
    "i don't have information",
    "i do not have information",
    "unable to verify",
    "cannot verify",
    "not aware of any",
    "no such",
    "i'm not certain",
    "i am not certain",
    "fictional",
    "does not exist",
    "doesn't exist",
]


def contains_hedge(response):
    response_lower = str(response).lower()
    return any(
        phrase in response_lower
        for phrase in HEDGE_PHRASES
    )


def apply_trap_heuristic(df):
    scored_df = df.copy()
    scored_df["hedge_detected"] = pd.NA

    mask = (
        scored_df["question_type"].eq("trap")
        & scored_df["api_status"].eq("success")
    )

    scored_df.loc[mask, "hedge_detected"] = (
        scored_df.loc[mask, "response"].apply(contains_hedge)
    )

    return scored_df

In [11]:
def judge_factual_answer(
    question,
    reference_answer,
    model_answer,
    judge_model=JUDGE_MODEL,
):
    prompt = f'''
Evaluate whether the model answer is factually correct.

Question:
{question}

Reference answer:
{reference_answer}

Model answer:
{model_answer}

Labels:
- correct: contains the essential facts and no important factual errors
- partially_correct: partly correct but incomplete, imprecise, or contains a minor error
- incorrect: contradicts or misses the central answer, or contains a major error

Evaluate factual content rather than writing style.
Return JSON with keys "label" and "justification".
'''.strip()

    try:
        response = client.chat.completions.create(
            model=judge_model,
            temperature=0,
            max_completion_tokens=200,
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are a strict factual-answer evaluator. "
                        "Return only valid JSON."
                    ),
                },
                {"role": "user", "content": prompt},
            ],
        )

        judgment = json.loads(
            response.choices[0].message.content
        )
        valid_labels = {
            "correct",
            "partially_correct",
            "incorrect",
        }
        label = judgment.get("label")

        if label not in valid_labels:
            raise ValueError(f"Invalid judge label: {label}")

        return {
            "judge_label": label,
            "judge_justification": judgment.get(
                "justification", ""
            ),
            "judge_status": "success",
        }

    except Exception as exc:
        return {
            "judge_label": "judge_error",
            "judge_justification": str(exc),
            "judge_status": "error",
        }

In [12]:
def apply_factual_judge(df):
    evaluated_df = df.copy()
    evaluated_df["judge_label"] = pd.NA
    evaluated_df["judge_justification"] = pd.NA
    evaluated_df["judge_status"] = "not_applicable"

    rows_to_judge = evaluated_df[
        evaluated_df["question_type"].eq("control")
        & evaluated_df["api_status"].eq("success")
    ]

    for index, row in rows_to_judge.iterrows():
        judgment = judge_factual_answer(
            question=row["question"],
            reference_answer=row["reference_answer"],
            model_answer=row["response"],
        )

        for key, value in judgment.items():
            evaluated_df.at[index, key] = value

    return evaluated_df

In [17]:
# PAID JUDGE CELL
baseline_scored_df = apply_trap_heuristic(baseline_df)
baseline_scored_df = apply_factual_judge(baseline_scored_df)

In [18]:
def make_control_summary(df):
    valid = df[
        df["question_type"].eq("control")
        & df["judge_status"].eq("success")
    ]

    summary = (
        valid.groupby(["experiment", "model"])["judge_label"]
        .value_counts()
        .rename("count")
        .reset_index()
    )

    summary["percentage"] = (
        summary["count"]
        / summary.groupby(["experiment", "model"])["count"].transform("sum")
        * 100
    )
    return summary


def make_trap_summary(df):
    valid = df[
        df["question_type"].eq("trap")
        & df["api_status"].eq("success")
    ].copy()

    valid["heuristic_label"] = valid["hedge_detected"].map({
        True: "hedge_detected",
        False: "no_hedge_detected",
    })

    summary = (
        valid.groupby(["experiment", "model"])["heuristic_label"]
        .value_counts()
        .rename("count")
        .reset_index()
    )

    summary["percentage"] = (
        summary["count"]
        / summary.groupby(["experiment", "model"])["count"].transform("sum")
        * 100
    )
    return summary


display(make_control_summary(baseline_scored_df))
display(make_trap_summary(baseline_scored_df))

,experiment,model,judge_label,count,percentage
0,baseline,gpt-4o,correct,9,90.0
1,baseline,gpt-4o,partially_correct,1,10.0
2,baseline,gpt-4o-mini,correct,9,90.0
3,baseline,gpt-4o-mini,partially_correct,1,10.0
4,baseline,gpt-5.6-luna,correct,10,100.0
5,baseline,gpt-5.6-sol,correct,10,100.0


,experiment,model,heuristic_label,count,percentage
0,baseline,gpt-4o,no_hedge_detected,10,100.0
1,baseline,gpt-4o-mini,no_hedge_detected,10,100.0
2,baseline,gpt-5.6-luna,no_hedge_detected,10,100.0
3,baseline,gpt-5.6-sol,no_hedge_detected,10,100.0


####Uncertainty-aware prompt

This condition tests whether explicitly instructing the model to acknowledge
uncertainty reduces unsupported answers. It must also be checked for unnecessary
hedging on factual control questions.

In [19]:
UNCERTAINTY_SYSTEM_PROMPT = """
You are a knowledgeable assistant. Answer the user's question
directly and concisely in 2–4 sentences. If you are not confident
that something in the question is real or accurate, say so clearly
instead of guessing.
""".strip()

In [ ]:
uncertainty_df = run_experiment(
    dataset=hallucination_dataset,
    models=MODELS_TO_COMPARE,
    system_prompt=UNCERTAINTY_SYSTEM_PROMPT,
    experiment_name="uncertainty_instruction",
    n_runs=N_RUNS,
)

In [21]:
from google.colab import files
uploaded_2 = files.upload()

Saving hallucination_uncertainty_raw.csv to hallucination_uncertainty_raw (1).csv


In [25]:
for filename, content in uploaded_2.items():
    uncertainty_df = pd.read_csv(io.BytesIO(content))
    break # Process only the first uploaded file

In [27]:
# PAID JUDGE CELL
uncertainty_scored_df = apply_trap_heuristic(uncertainty_df)
uncertainty_scored_df = apply_factual_judge(uncertainty_scored_df)


In [28]:
all_hallucination_results = pd.concat(
    [baseline_scored_df, uncertainty_scored_df],
    ignore_index=True,
)

display(make_control_summary(all_hallucination_results))
display(make_trap_summary(all_hallucination_results))

,experiment,model,judge_label,count,percentage
0,baseline,gpt-4o,correct,9,90.0
1,baseline,gpt-4o,partially_correct,1,10.0
2,baseline,gpt-4o-mini,correct,9,90.0
3,baseline,gpt-4o-mini,partially_correct,1,10.0
4,baseline,gpt-5.6-luna,correct,10,100.0
5,baseline,gpt-5.6-sol,correct,10,100.0
6,uncertainty_instruction,gpt-4o,correct,10,100.0
7,uncertainty_instruction,gpt-4o-mini,correct,9,90.0
8,uncertainty_instruction,gpt-4o-mini,partially_correct,1,10.0
9,uncertainty_instruction,gpt-5.6-luna,correct,10,100.0


,experiment,model,heuristic_label,count,percentage
0,baseline,gpt-4o,no_hedge_detected,10,100.0
1,baseline,gpt-4o-mini,no_hedge_detected,10,100.0
2,baseline,gpt-5.6-luna,no_hedge_detected,10,100.0
3,baseline,gpt-5.6-sol,no_hedge_detected,10,100.0
4,uncertainty_instruction,gpt-4o,no_hedge_detected,9,90.0
5,uncertainty_instruction,gpt-4o,hedge_detected,1,10.0
6,uncertainty_instruction,gpt-4o-mini,no_hedge_detected,10,100.0
7,uncertainty_instruction,gpt-5.6-luna,no_hedge_detected,9,90.0
8,uncertainty_instruction,gpt-5.6-luna,hedge_detected,1,10.0
9,uncertainty_instruction,gpt-5.6-sol,no_hedge_detected,9,90.0


### Limitations

- Each question was sampled only once.
- Temperature 1.0 introduces randomness.
- The question set is small and partly AI-generated.
- Keyword heuristics do not establish factual correctness.
- Fake questions should be independently verified as fake.
- An LLM judge may introduce model or provider bias.
- The results are exploratory rather than statistically conclusive.

###Next Steps

Future work should repeat every question at least five times with identical settings. This would produce 400 responses per prompt condition across four models and allow analysis of variation between runs.

### Overconfidence and Caliberation Failure

LLMs fail with overconfidence and calibration because they rarely express uncertainty and often present incorrect assertions with authority

**How to Test?**

Prepare 20 questions
- 10 10 straightforward questions for which the model is expected to have strong knowledge.
- 10 difficult, ambiguous, obscure, or potentially unanswerable questions.

For every question, the model should
1. provide an answer
2. Report a confidence score from 0% to 100%.
3. Briefly explain the basis for its answer.

**Set Up**

Each question will be asked five times, producing 100 responses in total. Every run will use GPT-4o mini at temperature 1.0.

**How to Measure**

1. Correctness
2. Self reported confidence
3. Lingusitic confidence

**Analysis**

Compare confidence with correctness

- Overall accuracy
- Average confidence
- Average confidence for correct responses
- Average confidence for incorrect responses
- Number and percentage of incorrect responses given with at least 80% confidence
    - Differences between easy and difficult questions
- Variation across the five repetitions of each question
- Difference between self-reported confidence and judged linguistic confidence


## Input and Context Failures

Input and Context Failures arise from the brittleness of the prompt and context interface which lead to lead to performance instability independent of model quality. These include:
1. Ambiguous or incomplete prompts
2. Prompt injection & adverserial inputs
3. Loss of context & truncation
4. Domain mismatch / out of distribution inputs
5. Conflicting or overlapping instructions

## System and Operational Failure

System and operational faults arise post-generation in the compatibility-rich application space. These include
1. Tool / API Invocation errors
2. External tool failure & runtime breakdowns
3. Communication breakdowns in multi-agent workflows
4. Misalignment with application logic and business rules
5. Cost-driven degradation and accuracy trade-offs